<a href="https://colab.research.google.com/github/yoohyunseok/Machine-Learnig-Deep-Learning/blob/main/homework4_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install featuretools

In [ ]:
import pandas as pd
import numpy as np
import featuretools as ft

clients = pd.read_csv('/clients.csv', parse_dates = ['joined'])
loans = pd.read_csv('/loans.csv', parse_dates = ['loan_start', 'loan_end'])
payments = pd.read_csv('/payments.csv', parse_dates = ['payment_date'])

# Create new entityset
es = ft.EntitySet(id = 'clients')

In [ ]:
print(clients.head())

   client_id     joined  income  credit_score
0      46109 2002-04-16  172677           527
1      49545 2007-11-14  104564           770
2      41480 2013-03-11  122607           585
3      46180 2001-11-06   43851           562
4      25707 2006-10-06  211422           621


In [ ]:
print(loans.head())

   client_id loan_type  loan_amount  repaid  loan_id loan_start   loan_end  \
0      46109      home        13672       0    10243 2002-04-16 2003-12-20   
1      46109    credit         9794       0    10984 2003-10-21 2005-07-17   
2      46109      home        12734       1    10990 2006-02-01 2007-07-05   
3      46109      cash        12518       1    10596 2010-12-08 2013-05-05   
4      46109    credit        14049       1    11415 2010-07-07 2012-05-21   

   rate  
0  2.15  
1  1.25  
2  0.68  
3  1.24  
4  3.13  


In [ ]:
print(payments.head())

   loan_id  payment_amount payment_date  missed
0    10243            2369   2002-05-31       1
1    10243            2439   2002-06-18       1
2    10243            2662   2002-06-29       0
3    10243            2268   2002-07-20       0
4    10243            2027   2002-07-31       1


In [ ]:
# Create an entity from the client dataframe
# This dataframe already has an index and a time index
# index=RDB primary key
# time index= date_time feature
es.add_dataframe(dataframe_name = 'clients',
dataframe = clients, index = 'client_id', time_index =
'joined')

Entityset: clients
  DataFrames:
    clients [Rows: 25, Columns: 4]
  Relationships:
    No relationships

In [ ]:
print(es)

Entityset: clients
  DataFrames:
    clients [Rows: 25, Columns: 4]
  Relationships:
    No relationships


In [ ]:
#Create an entity from the client dataframe
es.add_dataframe(dataframe_name= 'loans',
dataframe = loans, index = 'loan_id')

/usr/local/lib/python3.12/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/local/lib/python3.12/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(


Entityset: clients
  DataFrames:
    clients [Rows: 25, Columns: 4]
    loans [Rows: 443, Columns: 8]
  Relationships:
    No relationships

In [ ]:
# Create an entity from the payments dataframe
# This does not have an index
es.add_dataframe(dataframe_name = 'payments',
  dataframe = payments,
  logical_types = {'missed': 'Categorical'},
  make_index = True,
  index = 'payment_id',
  time_index = 'payment_date')

Entityset: clients
  DataFrames:
    clients [Rows: 25, Columns: 4]
    loans [Rows: 443, Columns: 8]
    payments [Rows: 3456, Columns: 5]
  Relationships:
    No relationships

In [ ]:
#check dataframes
print(es['clients'].head())
print(es['loans'].head())
print(es['payments'].head())

       client_id     joined  income  credit_score
42320      42320 2000-04-27  229481           563
39384      39384 2000-06-18  191204           617
26945      26945 2000-11-26  214516           806
41472      41472 2001-11-06  152214           638
46180      46180 2001-11-06   43851           562
       client_id loan_type  loan_amount  repaid  loan_id loan_start  \
10243      46109      home        13672       0    10243 2002-04-16   
10984      46109    credit         9794       0    10984 2003-10-21   
10990      46109      home        12734       1    10990 2006-02-01   
10596      46109      cash        12518       1    10596 2010-12-08   
11415      46109    credit        14049       1    11415 2010-07-07   

        loan_end  rate  
10243 2003-12-20  2.15  
10984 2005-07-17  1.25  
10990 2007-07-05  0.68  
10596 2013-05-05  1.24  
11415 2012-05-21  3.13  
      payment_id  loan_id  payment_amount payment_date missed
2113        2113    11988            2053   2000-03-05      0

In [ ]:
# Group loans by client id and calculate mean, max, min of loans
stats = loans.groupby('client_id')['loan_amount'].agg(['sum'])
stats.columns = ['total_loan_amount']
# Merge with the clients dataframe
stats = clients.merge(stats, left_on = 'client_id', right_index=True, how = 'left')
print(stats.head(10))

       client_id     joined  income  credit_score  total_loan_amount
42320      42320 2000-04-27  229481           563             105931
39384      39384 2000-06-18  191204           617             149444
26945      26945 2000-11-26  214516           806             106889
41472      41472 2001-11-06  152214           638             120173
46180      46180 2001-11-06   43851           562             154017
46109      46109 2002-04-16  172677           527             179032
32885      32885 2002-05-13   58955           642             148806
29841      29841 2002-08-17   38354           523             176634
38537      38537 2002-10-21  127183           643             152768
35214      35214 2003-08-08   95849           696             129124


In [ ]:
# Create a relationship between clients and loans
# Using es.add_relationship
es = es.add_relationship(parent_dataframe_name='clients',
                         parent_column_name='client_id',
                         child_dataframe_name='loans',
                         child_column_name='client_id')

# Relationship between previous loans and previous payments
es = es.add_relationship(parent_dataframe_name='loans',
                         parent_column_name='loan_id',
                         child_dataframe_name='payments',
                         child_column_name='loan_id')
print(es)

Entityset: clients
  DataFrames:
    clients [Rows: 25, Columns: 4]
    loans [Rows: 443, Columns: 8]
    payments [Rows: 3456, Columns: 5]
  Relationships:
    loans.client_id -> clients.client_id
    payments.loan_id -> loans.loan_id


In [ ]:
# Create new features using specified primitives
features, feature_names = ft.dfs(entityset = es,
target_dataframe_name = 'clients',
agg_primitives = ['mean', 'max', 'percent_true', 'last'],
trans_primitives = ['month'])

/usr/local/lib/python3.12/dist-packages/featuretools/synthesis/dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  agg_primitives: ['percent_true']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)
/usr/local/lib/python3.12/dist-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x7d02e7728ae0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  ).agg(to_agg)
/usr/local/lib/python3.12/dist-packages/featuretools/computational_backends/feature_set_calculator.py:785: Fut

In [ ]:
print(feature_names)

[<Feature: income>, <Feature: credit_score>, <Feature: LAST(loans.loan_amount)>, <Feature: LAST(loans.loan_id)>, <Feature: LAST(loans.loan_type)>, <Feature: LAST(loans.rate)>, <Feature: LAST(loans.repaid)>, <Feature: MAX(loans.loan_amount)>, <Feature: MAX(loans.rate)>, <Feature: MAX(loans.repaid)>, <Feature: MEAN(loans.loan_amount)>, <Feature: MEAN(loans.rate)>, <Feature: MEAN(loans.repaid)>, <Feature: LAST(payments.missed)>, <Feature: LAST(payments.payment_amount)>, <Feature: LAST(payments.payment_id)>, <Feature: MAX(payments.payment_amount)>, <Feature: MEAN(payments.payment_amount)>, <Feature: MONTH(joined)>, <Feature: LAST(loans.MAX(payments.payment_amount))>, <Feature: LAST(loans.MEAN(payments.payment_amount))>, <Feature: LAST(loans.MONTH(loan_end))>, <Feature: LAST(loans.MONTH(loan_start))>, <Feature: MAX(loans.LAST(payments.payment_amount))>, <Feature: MAX(loans.LAST(payments.payment_id))>, <Feature: MAX(loans.MEAN(payments.payment_amount))>, <Feature: MEAN(loans.LAST(payments.pa